# VPINN inverse 1D — notebook entièrement réécrit

## 1. Analyse détaillée du notebook initial

### Problèmes structurels identifiés

1. **Phase 1 presque sans physique**  
   La première phase ajuste surtout `T(x)` aux observations avec une pénalisation de courbure. Cela donne parfois une belle interpolation visuelle, mais ce n’est pas forcément une bonne estimation de `T'` ni de `T''`, or toute l’inversion de `f` dépend précisément de ces dérivées.

2. **Changement de formulation entre les phases**  
   La phase 2 utilise une relation faible cohérente avec l’idée VPINN, puis la phase 3 repasse sur un résidu fort `T'' + f`. C’est souvent un très mauvais compromis dans un problème inverse bruité, car la contrainte forte est la plus instable numériquement.

3. **Sur-apprentissage possible de la partie forte**  
   Même si la loss data reste présente, la forme forte peut forcer le réseau à fabriquer des oscillations locales dans `T` pour corriger `f`, ce qui dégrade la stabilité.

4. **Peu de diagnostics utiles**  
   Le notebook ne surveille pas assez le conditionnement du solveur spectral, la norme des coefficients, les gradients, ni les meilleurs checkpoints.

5. **Autograd utilisé sur des dérivées d’ordre 2 comme objectif principal**  
   C’est exactement le point le plus fragile. Ce n’est pas faux mathématiquement, mais ce n’est pas la formulation la plus robuste pour entraîner.


## 2. Idée de la réécriture

La nouvelle version suit cette logique :

- imposer les conditions aux bords **exactement** par construction ;
- apprendre `T(x)` de manière stable à partir des données ;
- reconstruire une première approximation de `f(x)` par un **solve faible direct** ;
- faire un raffinement conjoint avec une **loss VPINN faible** ;
- garder la forme forte seulement comme **diagnostic final**, pas comme moteur principal d’optimisation.

C’est généralement plus stable et plus fidèle à l’esprit VPINN.


## 3. Notebook réécrit — contenu complet

### Cellule 1 — imports, seed, config

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import time
from dataclasses import dataclass, asdict

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_default_dtype(torch.float32)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = "./results_vpinn_inverse_1d_rewrite"
os.makedirs(RESULTS_DIR, exist_ok=True)

def free_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {DEVICE}")
print(f"Results dir     : {RESULTS_DIR}")

@dataclass
class Config:
    n_obs: int = 25
    noise_rel: float = 0.01
    n_eval: int = 500
    n_quad: int = 256
    k_test: int = 16
    k_source: int = 20
    ff_k: int = 8
    hidden: tuple[int, ...] = (64, 64, 64)
    p1_epochs: int = 4000
    p1_lr: float = 2e-3
    p1_w_data: float = 1e3
    p1_w_h2: float = 5e-4
    p1_w_h1: float = 1e-5
    p2_ridge: float = 1e-6
    p2_ridge_power: int = 2
    p3_epochs: int = 6000
    p3_lr_T: float = 5e-4
    p3_lr_f: float = 2e-3
    p3_w_data: float = 1e3
    p3_w_weak: float = 2e2
    p3_w_anchor: float = 1e1
    p3_w_f_reg: float = 1e-6
    p3_w_h2: float = 1e-4
    p3_grad_clip: float = 1.0
    use_lbfgs: bool = True
    lbfgs_steps: int = 40
    lbfgs_lr: float = 0.4
    lbfgs_max_iter_per_step: int = 20

CFG = Config()
print("Configuration:")
for k, v in asdict(CFG).items():
    print(f"  - {k:18s}: {v}")


### Cellule 2 — définition du problème et données

In [ ]:
def f_true_np(x: np.ndarray) -> np.ndarray:
    return np.sin(np.pi * x) + 0.5 * np.sin(3.0 * np.pi * x)

def T_true_np(x: np.ndarray) -> np.ndarray:
    return np.sin(np.pi * x) / np.pi**2 + 0.5 * np.sin(3.0 * np.pi * x) / (9.0 * np.pi**2)

x_obs_np = np.linspace(0.05, 0.95, CFG.n_obs, dtype=np.float32)
T_clean_np = T_true_np(x_obs_np)
noise_amp = CFG.noise_rel * np.max(np.abs(T_clean_np))
T_obs_np = T_clean_np + noise_amp * np.random.randn(CFG.n_obs).astype(np.float32)
noise_floor = float(noise_amp**2)

x_obs = torch.tensor(x_obs_np, device=DEVICE).unsqueeze(1)
T_obs = torch.tensor(T_obs_np, device=DEVICE).unsqueeze(1)

x_eval = torch.linspace(0.0, 1.0, CFG.n_eval, device=DEVICE).unsqueeze(1)
x_eval_np = x_eval.detach().cpu().squeeze().numpy()

print(f"n_obs       : {CFG.n_obs}")
print(f"noise_rel   : {CFG.noise_rel:.2%}")
print(f"noise_amp   : {noise_amp:.3e}")
print(f"noise_floor : {noise_floor:.3e}")

plt.figure(figsize=(8, 4))
plt.plot(x_eval_np, T_true_np(x_eval_np), label="T true", lw=2.5)
plt.scatter(x_obs_np, T_obs_np, color="tab:red", label="observations", zorder=5)
plt.grid(alpha=0.3)
plt.legend()
plt.title("Observed temperature data")
plt.xlabel("x")
plt.ylabel("T(x)")
plt.tight_layout()
plt.show()


### Cellule 3 — quadrature faible

In [ ]:
def gauss_legendre_01(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    xg, wg = np.polynomial.legendre.leggauss(n)
    xg = 0.5 * (xg + 1.0)
    wg = 0.5 * wg
    xq = torch.tensor(xg, dtype=torch.float32, device=device).unsqueeze(1)
    wq = torch.tensor(wg, dtype=torch.float32, device=device).unsqueeze(1)
    return xq, wq

x_quad, w_quad = gauss_legendre_01(CFG.n_quad, DEVICE)
k_test = torch.arange(1, CFG.k_test + 1, dtype=torch.float32, device=DEVICE).view(-1, 1)
phi_test = torch.sin(k_test * math.pi * x_quad.T)
dphi_test = k_test * math.pi * torch.cos(k_test * math.pi * x_quad.T)

print(f"x_quad shape : {tuple(x_quad.shape)}")
print(f"phi_test     : {tuple(phi_test.shape)}")
print(f"dphi_test    : {tuple(dphi_test.shape)}")


### Cellule 4 — modèles

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, k: int = 8) -> None:
        super().__init__()
        freqs = torch.arange(1, k + 1, dtype=torch.float32) * math.pi
        self.register_buffer("freqs", freqs)

    @property
    def out_dim(self) -> int:
        return 1 + 2 * len(self.freqs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([x, torch.sin(x * self.freqs), torch.cos(x * self.freqs)], dim=-1)

class FourierMLP(nn.Module):
    def __init__(self, hidden: tuple[int, ...], ff_k: int) -> None:
        super().__init__()
        self.ff = FourierFeatures(ff_k)
        dims = [self.ff.out_dim] + list(hidden) + [1]
        layers: list[nn.Module] = []
        for i in range(len(dims) - 1):
            lin = nn.Linear(dims[i], dims[i + 1])
            nn.init.xavier_uniform_(lin.weight)
            nn.init.zeros_(lin.bias)
            layers.append(lin)
            if i < len(dims) - 2:
                layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(self.ff(x))

class HardBCWrapper(nn.Module):
    def __init__(self, base: nn.Module) -> None:
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * (1.0 - x) * self.base(x)

class SineSeriesSource(nn.Module):
    def __init__(self, k_source: int) -> None:
        super().__init__()
        self.k_source = k_source
        self.a = nn.Parameter(torch.zeros(k_source, dtype=torch.float32))
        kpi = torch.arange(1, k_source + 1, dtype=torch.float32) * math.pi
        self.register_buffer("kpi", kpi)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        basis = torch.sin(x * self.kpi)
        return (basis * self.a).sum(dim=1, keepdim=True)

T_net = HardBCWrapper(FourierMLP(hidden=CFG.hidden, ff_k=CFG.ff_k)).to(DEVICE)
f_net = SineSeriesSource(k_source=CFG.k_source).to(DEVICE)

print(f"T_net params: {sum(p.numel() for p in T_net.parameters())}")
print(f"f_net params: {sum(p.numel() for p in f_net.parameters())}")


### Cellule 5 — dérivées et losses

In [ ]:
def grad_1d(y: torch.Tensor, x: torch.Tensor, create_graph: bool) -> torch.Tensor:
    return torch.autograd.grad(
        y,
        x,
        grad_outputs=torch.ones_like(y),
        create_graph=create_graph,
        retain_graph=True,
        only_inputs=True,
    )[0]

def d1(model: nn.Module, x_in: torch.Tensor, create_graph: bool = True) -> torch.Tensor:
    x = x_in.detach().clone().requires_grad_(True)
    y = model(x)
    return grad_1d(y, x, create_graph=create_graph)

def d2(model: nn.Module, x_in: torch.Tensor, create_graph: bool = True) -> torch.Tensor:
    x = x_in.detach().clone().requires_grad_(True)
    y = model(x)
    y_x = grad_1d(y, x, create_graph=True)
    y_xx = grad_1d(y_x, x, create_graph=create_graph)
    return y_xx

def source_basis_matrix(x: torch.Tensor, k_source: int) -> torch.Tensor:
    k = torch.arange(1, k_source + 1, dtype=torch.float32, device=x.device).view(1, -1)
    return torch.sin(x * k * math.pi)

def weak_residual_vector(T_model: nn.Module, f_model: nn.Module) -> torch.Tensor:
    dT = d1(T_model, x_quad, create_graph=True).T
    f_vals = f_model(x_quad).T
    lhs = (dT * dphi_test * w_quad.T).sum(dim=1)
    rhs = (f_vals * phi_test * w_quad.T).sum(dim=1)
    return lhs - rhs

def weak_loss(T_model: nn.Module, f_model: nn.Module) -> torch.Tensor:
    r = weak_residual_vector(T_model, f_model)
    return (r**2).mean()

def h1_seminorm_sq(T_model: nn.Module) -> torch.Tensor:
    dT = d1(T_model, x_quad, create_graph=True)
    return (w_quad * dT**2).sum()

def h2_seminorm_sq(T_model: nn.Module) -> torch.Tensor:
    d2T = d2(T_model, x_quad, create_graph=True)
    return (w_quad * d2T**2).sum()

def source_coeff_reg(f_model: SineSeriesSource, power: int = 2) -> torch.Tensor:
    k = torch.arange(1, f_model.k_source + 1, dtype=torch.float32, device=f_model.a.device)
    return ((k**power) * f_model.a.pow(2)).mean()

def strong_residual_rms(T_model: nn.Module, f_model: nn.Module, x: torch.Tensor) -> float:
    with torch.enable_grad():
        r = d2(T_model, x, create_graph=False) + f_model(x)
    return float(torch.sqrt((r**2).mean()).detach().cpu().item())


### Cellule 6 — logs

In [ ]:
class SimpleLogger:
    def __init__(self) -> None:
        self.history: dict[str, list[float]] = {}

    def push(self, **kwargs: float) -> None:
        for key, value in kwargs.items():
            self.history.setdefault(key, []).append(float(value))

def print_train_line(prefix: str, epoch: int, total_epochs: int, **metrics: float) -> None:
    chunks = [f"{prefix} [{epoch:5d}/{total_epochs:5d}]"]
    chunks.extend([f"{k}={v:.3e}" for k, v in metrics.items()])
    print(" | ".join(chunks))

logger_p1 = SimpleLogger()
logger_p3 = SimpleLogger()


### Cellule 7 — phase 1

In [ ]:
best_p1_state = None
best_p1_loss = float("inf")
optimizer_p1 = optim.Adam(T_net.parameters(), lr=CFG.p1_lr)

for epoch in range(1, CFG.p1_epochs + 1):
    optimizer_p1.zero_grad()
    T_pred_obs = T_net(x_obs)
    loss_data = ((T_pred_obs - T_obs)**2).mean()
    loss_h1 = h1_seminorm_sq(T_net)
    loss_h2 = h2_seminorm_sq(T_net)
    loss = CFG.p1_w_data * loss_data + CFG.p1_w_h1 * loss_h1 + CFG.p1_w_h2 * loss_h2
    loss.backward()
    torch.nn.utils.clip_grad_norm_(T_net.parameters(), max_norm=1.0)
    optimizer_p1.step()
    logger_p1.push(loss=loss.item(), data=loss_data.item(), h1=loss_h1.item(), h2=loss_h2.item())
    if loss.item() < best_p1_loss:
        best_p1_loss = float(loss.item())
        best_p1_state = {k: v.detach().cpu().clone() for k, v in T_net.state_dict().items()}
    if epoch == 1 or epoch % 500 == 0 or epoch == CFG.p1_epochs:
        print_train_line("P1", epoch, CFG.p1_epochs, loss=loss.item(), data=loss_data.item(), h2=loss_h2.item())

if best_p1_state is not None:
    T_net.load_state_dict(best_p1_state)

with torch.no_grad():
    T_p1_np = T_net(x_eval).detach().cpu().squeeze().numpy()
T_p1_l2 = float(np.sqrt(np.mean((T_p1_np - T_true_np(x_eval_np))**2)))
print(f"Phase 1 temperature L2 error = {T_p1_l2:.6e}")


### Cellule 8 — solve direct faible pour `f`

In [ ]:
def direct_source_recovery_from_T(T_model: nn.Module, k_source: int, ridge: float, ridge_power: int):
    with torch.no_grad():
        dT_quad = d1(T_model, x_quad, create_graph=False)
    Phi = source_basis_matrix(x_quad, k_source)
    k = torch.arange(1, k_source + 1, dtype=torch.float32, device=DEVICE).view(1, -1)
    dPhi = (k * math.pi) * torch.cos(x_quad * k * math.pi)
    M = (Phi * w_quad).T @ Phi
    b = (dT_quad * dPhi * w_quad).sum(dim=0)
    ridge_diag = torch.arange(1, k_source + 1, dtype=torch.float32, device=DEVICE) ** ridge_power
    A = M + ridge * torch.diag(ridge_diag)
    a0 = torch.linalg.solve(A, b)
    stats = {
        "cond_M": float(torch.linalg.cond(M).detach().cpu()),
        "cond_A": float(torch.linalg.cond(A).detach().cpu()),
        "a0_norm_l2": float(torch.norm(a0).detach().cpu()),
        "a0_max_abs": float(torch.max(torch.abs(a0)).detach().cpu()),
    }
    return a0, stats

a0, p2_stats = direct_source_recovery_from_T(T_net, CFG.k_source, CFG.p2_ridge, CFG.p2_ridge_power)
with torch.no_grad():
    f_net.a.copy_(a0)


### Cellule 9 — phase 3 VPINN faible

In [ ]:
a_anchor = a0.detach().clone()
optimizer_p3 = optim.Adam([
    {"params": T_net.parameters(), "lr": CFG.p3_lr_T},
    {"params": f_net.parameters(), "lr": CFG.p3_lr_f},
])
all_trainable = list(T_net.parameters()) + list(f_net.parameters())
best_p3_objective = float("inf")
best_p3_state = None

for epoch in range(1, CFG.p3_epochs + 1):
    optimizer_p3.zero_grad()
    T_pred_obs = T_net(x_obs)
    loss_data = ((T_pred_obs - T_obs)**2).mean()
    loss_weak = weak_loss(T_net, f_net)
    loss_anchor = ((f_net.a - a_anchor)**2).mean()
    loss_f_reg = source_coeff_reg(f_net, power=2)
    loss_h2 = h2_seminorm_sq(T_net)
    loss = (
        CFG.p3_w_data * loss_data
        + CFG.p3_w_weak * loss_weak
        + CFG.p3_w_anchor * loss_anchor
        + CFG.p3_w_f_reg * loss_f_reg
        + CFG.p3_w_h2 * loss_h2
    )
    loss.backward()
    grad_norm = float(torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=CFG.p3_grad_clip).detach().cpu())
    optimizer_p3.step()
    logger_p3.push(loss=loss.item(), data=loss_data.item(), weak=loss_weak.item(), anchor=loss_anchor.item(), f_reg=loss_f_reg.item(), h2=loss_h2.item(), grad=grad_norm)
    if loss.item() < best_p3_objective:
        best_p3_objective = float(loss.item())
        best_p3_state = {
            "T_net": {k: v.detach().cpu().clone() for k, v in T_net.state_dict().items()},
            "f_net": {k: v.detach().cpu().clone() for k, v in f_net.state_dict().items()},
        }
    if epoch == 1 or epoch % 500 == 0 or epoch == CFG.p3_epochs:
        print_train_line("P3", epoch, CFG.p3_epochs, loss=loss.item(), data=loss_data.item(), weak=loss_weak.item(), anchor=loss_anchor.item(), grad=grad_norm)

if best_p3_state is not None:
    T_net.load_state_dict(best_p3_state["T_net"])
    f_net.load_state_dict(best_p3_state["f_net"])


### Cellule 10 — LBFGS final

In [ ]:
if CFG.use_lbfgs:
    params = list(T_net.parameters()) + list(f_net.parameters())
    optimizer_lbfgs = optim.LBFGS(params, lr=CFG.lbfgs_lr, max_iter=CFG.lbfgs_max_iter_per_step, history_size=50, line_search_fn="strong_wolfe")
    lbfgs_hist = []
    for step in range(1, CFG.lbfgs_steps + 1):
        def closure():
            optimizer_lbfgs.zero_grad()
            T_pred_obs = T_net(x_obs)
            loss_data = ((T_pred_obs - T_obs)**2).mean()
            loss_weak = weak_loss(T_net, f_net)
            loss_anchor = ((f_net.a - a_anchor)**2).mean()
            loss_f_reg = source_coeff_reg(f_net, power=2)
            loss_h2 = h2_seminorm_sq(T_net)
            loss = (
                CFG.p3_w_data * loss_data
                + CFG.p3_w_weak * loss_weak
                + CFG.p3_w_anchor * loss_anchor
                + CFG.p3_w_f_reg * loss_f_reg
                + CFG.p3_w_h2 * loss_h2
            )
            loss.backward()
            return loss
        loss_value = optimizer_lbfgs.step(closure)
        lbfgs_hist.append(float(loss_value.detach().cpu().item() if torch.is_tensor(loss_value) else loss_value))


### Cellule 11 — évaluation finale

In [ ]:
with torch.no_grad():
    T_final_np = T_net(x_eval).detach().cpu().squeeze().numpy()
    f_final_np = f_net(x_eval).detach().cpu().squeeze().numpy()

T_l2 = float(np.sqrt(np.mean((T_final_np - T_true_np(x_eval_np))**2)))
f_l2 = float(np.sqrt(np.mean((f_final_np - f_true_np(x_eval_np))**2)))
weak_rms_final = float(torch.sqrt((weak_residual_vector(T_net, f_net)**2).mean()).detach().cpu().item())
strong_rms_final = strong_residual_rms(T_net, f_net, x_eval)

print(f"T L2 error          : {T_l2:.6e}")
print(f"f L2 error          : {f_l2:.6e}")
print(f"Weak residual RMS   : {weak_rms_final:.6e}")
print(f"Strong residual RMS : {strong_rms_final:.6e}")


### Cellule 12 — plots et diagnostics

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(x_eval_np, T_true_np(x_eval_np), label="T true", lw=2.5)
plt.plot(x_eval_np, T_final_np, "--", label="T final", lw=2.0)
plt.scatter(x_obs_np, T_obs_np, label="observations", zorder=5)
plt.grid(alpha=0.3)
plt.legend()
plt.title(f"Temperature reconstruction | L2={T_l2:.3e}")

plt.subplot(1, 2, 2)
plt.plot(x_eval_np, f_true_np(x_eval_np), label="f true", lw=2.5)
plt.plot(x_eval_np, f_final_np, "--", label="f final", lw=2.0)
plt.grid(alpha=0.3)
plt.legend()
plt.title(f"Source reconstruction | L2={f_l2:.3e}")
plt.tight_layout()
plt.show()


## 4. Recommandations très concrètes pour améliorer encore les résultats

Les améliorations les plus rentables à tester ensuite sont :

1. **Augmenter légèrement `k_test`** vers 20–24 si la loss faible stagne.
2. **Réduire `k_source`** si tu vois des oscillations parasites dans `f`.
3. **Ne pas surpondérer `p3_w_weak` trop tôt** si les données sont très bruitées.
4. **Garder `p3_w_anchor` non nul** : c’est un excellent stabilisateur dans ce problème.
5. **Utiliser la forme forte seulement pour diagnostiquer**, pas pour entraîner.
6. **Surveiller les coefficients `a_k`** : si les hautes fréquences explosent, il faut augmenter `p3_w_f_reg` ou baisser `k_source`.


## 5. Verdict sur ton notebook initial

Le souci principal n’est pas que le code soit “mauvais” au sens syntaxique. Le vrai problème est que la stratégie d’optimisation mélange une bonne idée VPINN en phase 2 avec une phase 3 beaucoup plus fragile numériquement. C’est vraisemblablement **la** raison principale pour laquelle les résultats ne sont “pas fous”.

La version ci-dessus est plus cohérente, plus stable et plus défendable pour un problème inverse 1D.
